In [12]:
import os
import json
import math
import random
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms


**Dataclass Config**

- Writing a dataclass `Config`, which will contain data like directory paths, and other training or evaluation paramaters. 

In [9]:
@dataclass
class Config:
    data_dir: str = "/mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/learn-by-rebuilding/dlcv/cam-what-cnn-is-looking-at/data/"
    train_dir: str = "train"
    valid_dir: str = "valid"
    test_dir: str = "test"
    image_size: int = 224
    batch_size: int = 32
    num_workers: int = 4
    num_epochs: int = 5
    learning_rate: float = 1e-4
    weight_decay: float = 1e-4
    num_classes: int = 100
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    model_save_path: str = "resnet50_cam_best.pt"
    history_save_path: str = "training_history.json"
    plot_save_dir: str = "plots"
    seed: int = 42
    freeze_backbone: bool = False
    use_amp: bool = True


In [11]:
config = Config()
config

Config(data_dir='/mnt/c/Users/Rakesh-PC/Documents/1_GitHubSync_SSH/learn-by-rebuilding/dlcv/cam-what-cnn-is-looking-at/data/', train_dir='train', valid_dir='valid', test_dir='test', image_size=224, batch_size=32, num_workers=4, num_epochs=5, learning_rate=0.0001, weight_decay=0.0001, num_classes=100, device='cuda', model_save_path='resnet50_cam_best.pt', history_save_path='training_history.json', plot_save_dir='plots', seed=42, freeze_backbone=False, use_amp=True)

**Reproducibility**
- Writing a function to set seed for reproducibility later

In [13]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


**Data Transformation**

In [14]:
def get_data_transforms(image_size: int = 224):
    # ImageNet normalization for pretrained ResNet50
    imagenet_mean = [0.485, 0.456, 0.406]
    imagenet_std = [0.229, 0.224, 0.225]
    train_tfms = transforms.Compose([
        transforms.RandomResizedCrop(image_size, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.05),
        transforms.ToTensor(),
        transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
    ])
    eval_tfms = transforms.Compose([
        transforms.Resize(int(image_size * 1.14)),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
    ])
    # For CAM visualization: tensor transform without normalization will be helpful
    cam_vis_tfms = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
    ])
    return train_tfms, eval_tfms, cam_vis_tfms


**Dataset Builder**

In [15]:
def build_datasets(data_dir: str, image_size: int = 224):
    train_tfms, eval_tfms, cam_vis_tfms = get_data_transforms(image_size)
    train_path = Path(data_dir) / "train"
    valid_path = Path(data_dir) / "valid"
    test_path = Path(data_dir) / "test"
    train_dataset = datasets.ImageFolder(train_path, transform=train_tfms)
    valid_dataset = datasets.ImageFolder(valid_path, transform=eval_tfms)
    test_dataset = datasets.ImageFolder(test_path, transform=eval_tfms)
    # For CAM / raw image fetching, keep a parallel dataset with non-normalized tensor transforms
    cam_dataset = datasets.ImageFolder(test_path, transform=cam_vis_tfms)
    return train_dataset, valid_dataset, test_dataset, cam_dataset


**Dataset Loader**

In [16]:
def build_dataloaders(
    data_dir: str,
    batch_size: int = 32,
    num_workers: int = 4,
    image_size: int = 224,
):
    train_dataset, valid_dataset, test_dataset, cam_dataset = build_datasets(data_dir, image_size)
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
    )
    valid_loader = DataLoader(
        valid_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )
    cam_loader = DataLoader(
        cam_dataset,
        batch_size=1,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )
    class_names = train_dataset.classes
    class_to_idx = train_dataset.class_to_idx
    return train_loader, valid_loader, test_loader, cam_loader, class_names, class_to_idx
